# Simple Strategy Backtest

Demonstrates a basic moving average crossover strategy using AlphaTA.

In [ ]:
import numpy as np
import pandas as pd
import alpha_ta as ta

In [ ]:
# Generate synthetic OHLCV data
np.random.seed(42)
n = 500
close = np.cumsum(np.random.randn(n) * 0.5) + 100.0
high = close + np.abs(np.random.randn(n) * 0.3)
low = close - np.abs(np.random.randn(n) * 0.3)
open_ = close + np.random.randn(n) * 0.1
volume = np.random.uniform(1000, 5000, n)

df = pd.DataFrame({
    'open': open_, 'high': high, 'low': low,
    'close': close, 'volume': volume
})
df.head()

In [ ]:
# Compute fast and slow moving averages
fast_ma = np.array(ta.sma(close, timeperiod=10))
slow_ma = np.array(ta.sma(close, timeperiod=30))

# Generate signals: 1 = long, -1 = short, 0 = no position
signals = np.where(fast_ma > slow_ma, 1, -1)
signals[:30] = 0  # No signal until slow MA is ready

# Calculate returns
returns = np.diff(close) / close[:-1]
strategy_returns = signals[:-1] * returns

cumulative = np.cumprod(1 + strategy_returns)
print(f"Final portfolio value: {cumulative[-1]:.4f}")
print(f"Total return: {(cumulative[-1] - 1) * 100:.2f}%")
print(f"Sharpe ratio: {np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(252):.2f}")

In [ ]:
# Add RSI filter: only take long signals when RSI < 70
rsi = np.array(ta.rsi(close, timeperiod=14))
filtered_signals = signals.copy()
filtered_signals[(signals == 1) & (rsi > 70)] = 0

filtered_returns = filtered_signals[:-1] * returns
filtered_cumulative = np.cumprod(1 + filtered_returns)
print(f"Filtered strategy return: {(filtered_cumulative[-1] - 1) * 100:.2f}%")